In [1]:
# specifying the static variables
topic_name = "" # the kafka topic-name, i used player-data-v1
bootstrap_server = "" # bootstrap server name as per kafka configurations

table_name = "" # mysql table name, i used players_raw_data_v1
database_name = "" # mysql database name, i used test_db
user = "" # mysql conf, i used root
password = "" # mysql login password, i used root
hostname = "" # localstacks endpoint or localhost or IP address, i used IPv4 address
port_number = "" # mysql port number i used 3307, as i had already a running instance of mysql on port number 3306

bucket_name = "" # s3 bucket name
data_folder_name = f"{bucket_name}-data"
checkpoint_folder_name = f"{bucket_name}-cp-dir"

access_key = '' # aws_access_key_id of your localstack
secret_key = '' # aws_secret_access_key of your localstack

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Stream Data from Kafka to MySql and S3") 
    .config("spark.streaming.stopGracefullyOnShutdown", "true") 
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,mysql:mysql-connector-java:8.0.32,org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk:1.12.780,com.google.guava:guava:30.1.1-jre,org.apache.httpcomponents:httpcore:4.4.14,com.google.inject:guice:4.2.2,com.google.inject.extensions:guice-servlet:4.2.2")
    .config('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('fs.s3a.access.key', access_key)
    .config('fs.s3a.secret.key', secret_key)
    .config('spark.hadoop.fs.s3a.endpoint', f'http://{hostname}:4566')
    .config('spark.hadoop.fs.s3a.connection.maximum', '100')
    .config('spark.hadoop.fs.s3a.fast.upload', 'true')
    .config('spark.hadoop.fs.s3a.fast.upload.buffer', 'bytebuffer')
    .config('spark.hadoop.fs.s3a.fast.upload.active.blocks', '4')
    .config('spark.hadoop.fs.s3a.readahead.range', '64K')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]") 
    .getOrCreate()
)

In [3]:
spark

In [4]:
input_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_server)
    .option("subscribe", topic_name)
    .option("startingOffsets", "earliest")
    .load()
)

In [5]:
input_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
from pyspark.sql.functions import expr

kafka_json_df = input_df.withColumn("value", expr("CAST(value AS STRING)"))

In [7]:
kafka_json_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [8]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

json_schema = StructType([
    StructField("player_id", StringType(), True),
    StructField("name_info", StructType([
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True)
    ]), True),
    StructField("nationality", StringType(), True),
    StructField("clubs", ArrayType(StringType()), True),
    StructField("role", StringType(), True)
])

In [9]:
from pyspark.sql.functions import from_json, col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [10]:
streaming_df.printSchema()

root
 |-- player_id: string (nullable = true)
 |-- name_info: struct (nullable = true)
 |    |-- first_name: string (nullable = true)
 |    |-- last_name: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- clubs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- role: string (nullable = true)



In [11]:
from pyspark.sql.functions import col

streaming_df = streaming_df.select(
    col("player_id"),
    col("name_info.first_name"),
    col("name_info.last_name"),
    col("nationality"),
    col("clubs"),
    col("role"),
)

In [12]:
streaming_df.printSchema()

root
 |-- player_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- clubs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- role: string (nullable = true)



In [13]:
import boto3

# Boto3 session for S3
session = boto3.Session(
    aws_access_key_id='test',
    aws_secret_access_key='test',
    region_name='us-east-1'
)
s3 = session.client('s3', endpoint_url=f'http://{hostname}:4566')

In [15]:
from botocore.exceptions import ClientError 

def folder_exists(bucket_name, folder_name):
    """
    Check if the folder exists in the S3 bucket. In S3, folders are represented by objects with a trailing `/`.
    """
    # Ensure folder_name ends with '/'
    if not folder_name.endswith('/'):
        folder_name += '/'
    
    try:
        # Attempt to head the object to check if it exists
        response = s3.head_object(Bucket=bucket_name, Key=folder_name)
        # If the object exists, it's a folder
        return True
    except ClientError as e:
        # If the error is "Not Found", it's likely the folder doesn't exist
        if e.response['Error']['Code'] == '404':
            return False
        else:
            # Raise the exception if it's a different error
            raise

def create_folder(bucket_name, folder_name):
    """
    Creates a folder in the S3 bucket if it doesn't already exist.
    """
    if not folder_exists(bucket_name, folder_name):
        # Ensure folder_name ends with '/'
        if not folder_name.endswith('/'):
            folder_name += '/'
        
        # Create the folder by uploading an empty object with the folder name
        try:
            s3.put_object(Bucket=bucket_name, Key=folder_name)
            print(f'Folder "{folder_name}" created successfully in bucket "{bucket_name}"!')
        except ClientError as e:
            print(f'Error creating folder: {e}')
    else:
        print(f'Folder "{folder_name}" already exists in bucket "{bucket_name}".')


In [16]:
create_folder(bucket_name, data_folder_name)

Folder "players-v1-data/" created successfully in bucket "players-v1"!


In [19]:
create_folder(bucket_name, checkpoint_folder_name)

Folder "players-v1-cp-dir" already exists in bucket "players-v1".


In [20]:
from datetime import datetime
import time

def getDateInfo():
    now = datetime.now()
    curr_time = time.strftime("%H:%M:%S", time.localtime())
    return now.year, now.month, now.day, now.hour, str(curr_time).replace(":", "_")
    
getDateInfo()

(2025, 1, 6, 8, '08_43_08')

In [25]:
import pandas as pd

def shareParquet(df, batch_id):
    print("Batch id: " + str(batch_id))
    
    yr, mn, dy, hr, tm = getDateInfo() 
    parquet_file_name = f"player_data_{tm}.parquet"
    parquet_file_path = f"/tmp/{parquet_file_name}"

    df = df.toPandas()
    df.to_parquet(parquet_file_path, engine='pyarrow')
    print(f"DataFrame saved as {parquet_file_name} at {parquet_file_path}")
    
    # Upload the Parquet file to S3
    try:
        s3.upload_file(parquet_file_path, bucket_name, f"{data_folder_name}/{parquet_file_name}")
        print(f"File {parquet_file_name} uploaded successfully to {data_folder_name}.")
    except NoCredentialsError:
        print("Credentials not available.")
    except Exception as e:
        print(f"Error uploading file: {e}")


In [26]:
def sendData(df, batch_id):
    
    # Writing to mysql table
#     mysql_url = f"jdbc:mysql://{hostname}:{port_number}/{database_name}"
#     mysql_properties = {
#         "user": user,
#         "password": password,
#         "driver": "com.mysql.cj.jdbc.Driver"
#     }
    
#     df.write.jdbc(url=mysql_url, table=table_name, mode="append", properties=mysql_properties)
#     print(f"Data written to MySQL table: {table_name}")
    
    # writing to s3
    s3Path = f"s3a://{bucket_name}/{data_folder_name}"
    df.write.mode("append").format("parquet").save(s3Path)
    print(f"Data written to s3 path: {s3Path}")
    
    df.show()
    

In [22]:
checkpoint_dir = f"s3a://{bucket_name}/{checkpoint_folder_name}"

In [ ]:
# Start streaming
(streaming_df
 .writeStream
 .foreachBatch(sendData)
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", checkpoint_dir)
 .start()
 .awaitTermination())

Data written to s3 path: s3a://players-v1/players-v1-data
+---------+----------+---------+-----------+--------------------+-----------+
|player_id|first_name|last_name|nationality|               clubs|       role|
+---------+----------+---------+-----------+--------------------+-----------+
|       30|      Gary|Steamboat|      Spain|[Kent, New South ...|All-Rounder|
|       20|      Mark|    Rowan|   Scotland|[Tasmania, New So...|     Bowler|
+---------+----------+---------+-----------+--------------------+-----------+

Data written to s3 path: s3a://players-v1/players-v1-data
+---------+----------+---------+-----------+--------------------+-------+
|player_id|first_name|last_name|nationality|               clubs|   role|
+---------+----------+---------+-----------+--------------------+-------+
|       73|       Joe|     Root|      Spain|[Tasmania, Surrey...| Bowler|
|       25|   Micheal|    Smith|   Scotland|[Queensland, Suss...|Batsman|
+---------+----------+---------+-----------+-